# Question 8 - VQ-GAN Straight-Through Estimator
### 1. Why is nearest-neighbour / argmax lookup non-differentiable?
$k^* = \arg\min_k \lVert z_e - e_k\rVert$ returns a **discrete index**. It is piecewise constant: small changes in $z_e$ do not change the chosen code, so the gradient is 0 almost everywhere and undefined at the boundaries. No useful gradient reaches the encoder.

### 2. STE (one line)
`z_q = z_e + tf.stop_gradient(z_q - z_e)`

Forward pass: the value equals $z_q$. Backward pass: the gradient is copied straight to $z_e$ (identity).

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

In [ ]:
class VectorQuantizer(layers.Layer):
    def __init__(self, num_codes=8, dim=4):
        super().__init__()
        self.codebook = self.add_weight(shape=(num_codes, dim), initializer="random_normal", name="codebook")

    def call(self, z_e):
        # squared distance to every code vector
        d = tf.reduce_sum(z_e ** 2, 1, keepdims=True) - 2 * tf.matmul(z_e, self.codebook, transpose_b=True) \
            + tf.reduce_sum(self.codebook ** 2, 1)
        idx = tf.argmin(d, axis=1)                  # non-differentiable
        z_q = tf.gather(self.codebook, idx)
        z_q_ste = z_e + tf.stop_gradient(z_q - z_e) # straight-through estimator
        return z_q_ste, z_q, idx

In [ ]:
vq = VectorQuantizer()
z_e = tf.Variable(tf.random.normal((5, 4)))

with tf.GradientTape(persistent=True) as tape:
    z_q_ste, z_q, idx = vq(z_e)
    loss_ste = tf.reduce_sum(z_q_ste)
    loss_raw = tf.reduce_sum(z_q)

print("chosen codes:", idx.numpy())
print("forward values equal z_q:", np.allclose(z_q_ste.numpy(), z_q.numpy()))
print("grad to z_e WITHOUT STE:", tape.gradient(loss_raw, z_e))
print("grad to z_e WITH STE   :\n", tape.gradient(loss_ste, z_e).numpy())